# Industrial Wastewater TCN — Spike-Aware Forecasting

Module 3: Improved TCN with project-specific temporal forecasting formulation.

## 1. Setup and Data Loading

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

In [ ]:
from modules.data_acquisition import load_dataset, validate_dataset

DATASET_PATH = PROJECT_ROOT / 'dataset' / 'wastewater_10000.csv'
df = load_dataset(str(DATASET_PATH))
validate_dataset(df)
print(f'Columns: {list(df.columns)}')
print(f'Shape: {df.shape}')

In [ ]:
from config import PARAMETERS
param_cols = [PARAMETERS[k] for k in PARAMETERS]
df[param_cols].describe().T

## 2. Preprocessing (Scaler fitted on train data only)

In [ ]:
from modules.preprocessing import (
    preprocess_data, create_sequences, split_chronological,
    save_scaler, StandardScaler
)
from config import LOOK_BACK, HORIZON, PARAM_KEYS

# Clean data
df_clean = preprocess_data(df)
param_cols_ordered = [PARAMETERS[k] for k in PARAM_KEYS]
raw_values = df_clean[param_cols_ordered].values

# Create sequences BEFORE scaling
X, y = create_sequences(raw_values)

# Split chronologically BEFORE scaling (prevents data leakage)
X_train, X_test, y_train, y_test = split_chronological(X, y)

# Fit scaler ONLY on training data
scaler = StandardScaler()
n_train = X_train.shape[0]
scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

# Scale train and test using the training-fitted scaler
X_train = scaler.transform(X_train.reshape(-1, 5)).reshape(X_train.shape)
X_test = scaler.transform(X_test.reshape(-1, 5)).reshape(X_test.shape)
y_train = scaler.transform(y_train.reshape(-1, 5)).reshape(y_train.shape)
y_test = scaler.transform(y_test.reshape(-1, 5)).reshape(y_test.shape)

save_scaler(scaler, str(PROJECT_ROOT / 'models' / 'scaler.pkl'))

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Scaler fitted on TRAINING data only (no test leakage).')
print(f'Scaler mean: {scaler.mean_}')
print(f'Scaler std:  {scaler.scale_}')

## 3. Baseline TCN (Original Architecture)

Architecture: 3 TCN blocks (dilation [1,2,4]), `x[:, -1, :]` bottleneck, Dense output head.

Loss: Standard MSE.

In [ ]:
from modules.tcn_prediction import build_tcn_model, train_tcn_model, evaluate_tcn_model
import tensorflow as tf

# Train/val split
val_split = 0.1
si = int(len(X_train) * (1 - val_split))
X_tr_base, X_val_base = X_train[:si], X_train[si:]
y_tr_base, y_val_base = y_train[:si], y_train[si:]
print(f'Baseline split: train={len(X_tr_base)}, val={len(X_val_base)}, test={len(X_test)}')

# Build baseline
model_base = build_tcn_model(input_shape=(LOOK_BACK, 5))

In [ ]:
history_base = train_tcn_model(
    model_base, X_tr_base, y_tr_base, X_val_base, y_val_base,
    epochs=40, batch_size=64, patience=10,
    model_path=str(PROJECT_ROOT / 'models' / 'tcn_model.keras')
)

print(f'\nEpochs trained: {len(history_base.history["loss"])}')
print(f'Best val_loss: {min(history_base.history["val_loss"]):.6f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_base.history['loss'], label='Train')
axes[0].plot(history_base.history['val_loss'], label='Val')
axes[0].set_title('Baseline — Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[1].plot(history_base.history['mae'], label='Train MAE')
axes[1].plot(history_base.history['val_mae'], label='Val MAE')
axes[1].set_title('Baseline — Model MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
results_base = evaluate_tcn_model(model_base, X_test, y_test)

## 4. Baseline — Prediction vs Actual

In [ ]:
from modules.tcn_prediction import predict_next_24h
import pandas as pd

# Evaluate across ALL test samples
n_test = X_test.shape[0]
all_pred_base = np.zeros((n_test, 24, 5))
all_true_orig = np.zeros((n_test, 24, 5))

for idx in range(n_test):
    all_pred_base[idx] = predict_next_24h(model_base, X_test[idx], scaler)
    all_true_orig[idx] = scaler.inverse_transform(y_test[idx])

print(f'Generated predictions for {n_test} test samples.')

In [ ]:
import numpy as np

param_labels = ['pH', 'COD (mg/L)', 'BOD (mg/L)', 'TDS (mg/L)', 'Temperature (°C)']

fig, axes = plt.subplots(5, 1, figsize=(12, 15), sharex=True)
for i, label in enumerate(param_labels):
    # Aggregate across test samples: mean prediction and mean actual
    mean_pred = np.mean(all_pred_base[:, :, i], axis=0)
    mean_true = np.mean(all_true_orig[:, :, i], axis=0)
    axes[i].plot(range(24), mean_pred, 'r-', label='Baseline Predicted', linewidth=2)
    axes[i].plot(range(24), mean_true, 'b--', label='Actual', linewidth=2)
    axes[i].set_ylabel(label)
    axes[i].set_title(f'{label} — Baseline 24h Prediction vs Actual (mean over test)')
    axes[i].legend()
axes[-1].set_xlabel('Hours Ahead')
plt.tight_layout()
plt.show()

## 5. Improved TCN — Spike-Aware Forecasting

### Forecasting Formulation

```
y_hat(t+h) = y(t) + alpha_h * T(t) + beta_h * A(t) + gamma_h * S(t)
```

| Component | Meaning | How computed |
|-----------|---------|---------------|
| `y(t)` | Latest observed value | Implicit in TCN features |
| `T(t)` | Temporal trend | `x(t) - x(t-1)` over input window |
| `A(t)` | Acceleration | `T(t) - T(t-1)` |
| `S(t)` | Spike signal | z-score of input vs input statistics |
| `alpha_h, beta_h, gamma_h` | Learned coefficients | via Dense layers |

### Architecture Changes

1. **4 TCN blocks** with dilation `[1, 2, 4, 8]` — receptive field = 30 (full 24h)
2. **Temporal decoder** with Conv1D — preserves all 24 timesteps (no `x[:, -1, :]`)
3. **Explicit T, A, S features** concatenated with input and output
4. **Spike-aware Huber loss** with adaptive sample weighting

In [ ]:
from modules.tcn_prediction import (
    build_tcn_model_v2, train_tcn_model_v2, evaluate_tcn_model,
    predict_next_24h_v2, evaluate_spikes, compare_metrics
)
import numpy as np

# Build improved model
model_v2 = build_tcn_model_v2(input_shape=(LOOK_BACK, 5))

In [ ]:
# Train/val split (same as baseline for fair comparison)
si = int(len(X_train) * (1 - val_split))
X_tr_v2, X_val_v2 = X_train[:si], X_train[si:]
y_tr_v2, y_val_v2 = y_train[:si], y_train[si:]

scaler_mean = scaler.mean_.astype(np.float32)
scaler_std = scaler.scale_.astype(np.float32)

model_v2, history_v2, sm, ss = train_tcn_model_v2(
    model_v2, X_tr_v2, y_tr_v2, X_val_v2, y_val_v2,
    scaler_mean=scaler_mean, scaler_std=scaler_std,
    epochs=60, batch_size=32, patience=15,
    model_path=str(PROJECT_ROOT / 'models' / 'tcn_model_v2.keras'),
    stats_path=str(PROJECT_ROOT / 'models' / 'scaler_stats.npz'),
)

print(f'\nEpochs trained: {len(history_v2.history["loss"])}')
print(f'Best val_loss: {min(history_v2.history["val_loss"]):.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_v2.history['loss'], label='Train')
axes[0].plot(history_v2.history['val_loss'], label='Val')
axes[0].set_title('Improved TCN v2 — Spike-Aware Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Spike-Aware Loss')
axes[0].legend()
axes[1].plot(history_v2.history['mae'], label='Train MAE')
axes[1].plot(history_v2.history['val_mae'], label='Val MAE')
axes[1].set_title('Improved TCN v2 — Model MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
results_v2 = evaluate_tcn_model(model_v2, X_test, y_test)

## 6. Improved Model — Prediction vs Actual

In [ ]:
all_pred_v2 = np.zeros((n_test, 24, 5))
for idx in range(n_test):
    all_pred_v2[idx] = predict_next_24h_v2(model_v2, X_test[idx], scaler)

fig, axes = plt.subplots(5, 1, figsize=(12, 15), sharex=True)
for i, label in enumerate(param_labels):
    mean_pred = np.mean(all_pred_v2[:, :, i], axis=0)
    mean_true = np.mean(all_true_orig[:, :, i], axis=0)
    axes[i].plot(range(24), mean_pred, 'r-', label='Improved Predicted', linewidth=2)
    axes[i].plot(range(24), mean_true, 'b--', label='Actual', linewidth=2)
    axes[i].set_ylabel(label)
    axes[i].set_title(f'{label} — Improved TCN v2 24h Prediction vs Actual')
    axes[i].legend()
axes[-1].set_xlabel('Hours Ahead')
plt.tight_layout()
plt.show()

## 7. Side-by-Side Comparison: Baseline vs Improved

In [ ]:
compare_metrics(results_base, results_v2)

In [ ]:
# Prediction comparison per parameter
fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=True)
for i, label in enumerate(param_labels):
    mean_pred_b = np.mean(all_pred_base[:, :, i], axis=0)
    mean_pred_v = np.mean(all_pred_v2[:, :, i], axis=0)
    mean_true = np.mean(all_true_orig[:, :, i], axis=0)
    axes[i].plot(range(24), mean_true, 'k--', label='Actual', linewidth=2.5, alpha=0.8)
    axes[i].plot(range(24), mean_pred_b, 'r-', label='Baseline', linewidth=2, alpha=0.7)
    axes[i].plot(range(24), mean_pred_v, 'g-', label='Improved v2', linewidth=2, alpha=0.7)
    axes[i].set_ylabel(label)
    axes[i].set_title(f'{label} — Baseline vs Improved vs Actual')
    axes[i].legend(loc='best')
axes[-1].set_xlabel('Hours Ahead')
plt.tight_layout()
plt.show()

## 8. Sample-Level Comparison (Random Test Samples)

In [ ]:
# Show a few specific test samples where spikes occur
# Find samples with high variance in actual values (likely spike periods)
sample_var = np.var(all_true_orig, axis=(1, 2))
spike_indices = np.argsort(sample_var)[-5:][::-1]

fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=True)
for i, label in enumerate(param_labels):
    # Aggregate across the high-variance samples
    for si_idx, si in enumerate(spike_indices):
        style = '-' if si_idx == 0 else ':'
        lw = 2 if si_idx == 0 else 1
        if i == 0:
            axes[i].plot(range(24), all_true_orig[si, :, i], 'b' + style,
                         linewidth=lw, alpha=0.5, label='Actual' if si_idx == 0 else '')
            axes[i].plot(range(24), all_pred_base[si, :, i], 'r' + style,
                         linewidth=lw, alpha=0.5, label='Baseline' if si_idx == 0 else '')
            axes[i].plot(range(24), all_pred_v2[si, :, i], 'g' + style,
                         linewidth=lw, alpha=0.5, label='Improved' if si_idx == 0 else '')
        else:
            axes[i].plot(range(24), all_true_orig[si, :, i], 'b' + style, linewidth=lw, alpha=0.5)
            axes[i].plot(range(24), all_pred_base[si, :, i], 'r' + style, linewidth=lw, alpha=0.5)
            axes[i].plot(range(24), all_pred_v2[si, :, i], 'g' + style, linewidth=lw, alpha=0.5)
    axes[i].set_ylabel(label)
    axes[i].set_title(f'{label} — High-variance samples (Baseline=red, Improved=green, Actual=blue)')
    axes[i].legend()
axes[-1].set_xlabel('Hours Ahead')
plt.tight_layout()
plt.show()

## 9. Spike Detection Evaluation

Using training-set statistics to define spike thresholds:
- A value is a **spike** if it deviates from the training mean by more than 2.5 standard deviations.
- **Window-level**: a 24h prediction window is flagged as a spike if ANY parameter at ANY timestep exceeds the threshold.
- This evaluates whether the model learns to predict anomalous values (not just averages).

In [ ]:
spike_results_base = evaluate_spikes(
    all_true_orig, all_pred_base, scaler_mean, scaler_std, spike_threshold=2.5
)

In [ ]:
spike_results_v2 = evaluate_spikes(
    all_true_orig, all_pred_v2, scaler_mean, scaler_std, spike_threshold=2.5
)

In [ ]:
# Spike detection comparison summary
print('\n' + '=' * 60)
print('SPIKE DETECTION COMPARISON')
print('=' * 60)
print(f'{"Metric":<25} {"Baseline":>12} {"Improved":>12}')
print('-' * 60)
for key in ['Actual spike windows', 'Predicted spike windows', 'True positives',
            'False positives', 'False negatives']:
    b = spike_results_base['overall'][key]
    v = spike_results_v2['overall'][key]
    print(f'  {key:<23} {b:>12} {v:>12}')
for key in ['Spike precision', 'Spike recall', 'Spike F1']:
    b = spike_results_base['overall'][key]
    v = spike_results_v2['overall'][key]
    print(f'  {key:<23} {b:>12.4f} {v:>12.4f}')
print('=' * 60)

## 10. Model Summary

In [ ]:
from modules.tcn_prediction import save_tcn_model

# Baseline is already saved. Save improved model.
save_tcn_model(model_v2, str(PROJECT_ROOT / 'models' / 'tcn_model_v2.keras'))
print('\nModels saved:')
print('  Baseline: models/tcn_model.keras')
print('  Improved: models/tcn_model_v2.keras')

## 11. Final Report

### Architecture Comparison

| Feature | Baseline | Improved v2 |
|---------|----------|-------------|
| TCN blocks | 3 (dilation [1,2,4]) | 4 (dilation [1,2,4,8]) |
| Receptive field | 14 timesteps | 30 timesteps (full 24h) |
| Temporal aggregation | `x[:, -1, :]` (last timestep only) | Full Conv1D temporal decoder |
| Output head | Dense from 64-dim vector | Dense from 842-dim (full temporal) |
| Loss function | Standard MSE | Spike-Aware Huber |
| Forecasting | End-to-end | Trend + Acceleration + Spike components |
| Parameters | ~67K | ~140K |

### Forecasting Formulation

```
y_hat(t+h) = y(t) + alpha_h * T(t) + beta_h * A(t) + gamma_h * S(t)
```

- **Trend T(t)**: computed as `x(t) - x(t-1)` over the 24h input window
- **Acceleration A(t)**: computed as `T(t) - T(t-1)`
- **Spike signal S(t)**: z-score of each timestep vs input statistics
- **alpha, beta, gamma**: learned by the network via Dense layers

### Key Design Decisions

1. No future values used — T, A, S derived from input window only
2. Spike-aware loss weights samples with extreme values higher (3x)
3. Full temporal information preserved through Conv1D decoder
4. Scaler fitted on training data only (no test leakage)